In [ ]:
import pandas as pd
import numpy as np

# Load the shared datasets
train = pd.read_csv("shared_data/train_clean.csv")
val = pd.read_csv("shared_data/val_clean.csv")
test = pd.read_csv("shared_data/test_clean.csv")

target = "is_canceled"

X_train = train.drop(columns=[target])
y_train = train[target]

X_val = val.drop(columns=[target])
y_val = val[target]

X_test = test.drop(columns=[target])
y_test = test[target]

# Check dataset shapes
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# Ensure consistent features
assert X_train.columns.equals(X_val.columns)
assert X_train.columns.equals(X_test.columns)

# Check missing values
assert X_train.isna().sum().sum() == 0
assert X_val.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

print("Data validation: PASSED")


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Always predict the most frequent class
baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_val)

print("Baseline Accuracy:",
      accuracy_score(y_val, baseline_pred))

print("Baseline Precision:",
      precision_score(y_val, baseline_pred, zero_division=0))

print("Baseline Recall:",
      recall_score(y_val, baseline_pred, zero_division=0))

print("Baseline F1:",
      f1_score(y_val, baseline_pred, zero_division=0))


In [ ]:
categorical_features = [
    "hotel",
    "arrival_date_month",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "deposit_type",
    "customer_type"
]

numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

print("Categorical:", len(categorical_features))
print("Numerical:", len(numerical_features))


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"),
     categorical_features)
])

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=2000,
        random_state=42
    ))
])

lr_pipeline.fit(X_train, y_train)

print("Model training completed!")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = lr_pipeline.predict(X_val)

print(classification_report(
    y_val, y_pred,
    target_names=["Not Canceled", "Canceled"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))


In [ ]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_score
from sklearn.metrics import recall_score, f1_score

results = []

for c in [0.01, 0.1, 1, 10, 100]:
    model = clone(lr_pipeline)
    model.set_params(classifier__C=c)

    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    results.append({
        "C": c,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred),
        "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred)
    })

tuning_results = pd.DataFrame(results)
print(tuning_results.round(4))

In [ ]:
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

model = clone(lr_pipeline)
model.set_params(classifier__C=10)
model.fit(X_train, y_train)

val_prob = model.predict_proba(X_val)[:, 1]

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
results = []

for threshold in thresholds:
    pred = (val_prob >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred),
        "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred)
    })

threshold_results = pd.DataFrame(results)
print(threshold_results.round(4))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(
    threshold_results["Threshold"],
    threshold_results["Precision"],
    marker="o",
    label="Precision"
)

plt.plot(
    threshold_results["Threshold"],
    threshold_results["Recall"],
    marker="s",
    label="Recall"
)

plt.plot(
    threshold_results["Threshold"],
    threshold_results["F1"],
    marker="^",
    label="F1"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Validation Performance by Threshold")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Fixed threshold selected on Validation
final_threshold = 0.30

# Predict probabilities on the untouched Test Set
test_prob = model.predict_proba(X_test)[:, 1]

# Apply the selected threshold
test_pred = (test_prob >= final_threshold).astype(int)

# Final evaluation
print("Accuracy:", accuracy_score(y_test, test_pred))
print("Precision:", precision_score(y_test, test_pred))
print("Recall:", recall_score(y_test, test_pred))
print("F1-score:", f1_score(y_test, test_pred))
print("ROC-AUC:", roc_auc_score(y_test, test_prob))

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))


In [ ]:
# ============================================================
# SAVE FINAL LOGISTIC REGRESSION RESULTS
# ============================================================

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)


# ------------------------------------------------------------
# 1. Create output folder
# ------------------------------------------------------------

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", output_dir.resolve())


# ------------------------------------------------------------
# 2. Final Logistic Regression configuration
# ------------------------------------------------------------

FINAL_C = 10
FINAL_THRESHOLD = 0.30

print("\nFinal model configuration:")
print("C =", FINAL_C)
print("Threshold =", FINAL_THRESHOLD)


# ------------------------------------------------------------
# 3. Final Logistic Regression test metrics
# ------------------------------------------------------------

lr_accuracy = accuracy_score(y_test, test_pred)
lr_precision = precision_score(y_test, test_pred)
lr_recall = recall_score(y_test, test_pred)
lr_f1 = f1_score(y_test, test_pred)
lr_roc_auc = roc_auc_score(y_test, test_prob)

final_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "C": FINAL_C,
        "Threshold": FINAL_THRESHOLD,
        "Accuracy": lr_accuracy,
        "Precision": lr_precision,
        "Recall": lr_recall,
        "F1": lr_f1,
        "ROC_AUC": lr_roc_auc
    }
])

final_results.to_csv(
    output_dir / "logistic_regression_final_results.csv",
    index=False
)

print("\n=== Final Logistic Regression Test Results ===")
print(final_results.round(4))


# ------------------------------------------------------------
# 4. Majority-class baseline on Test Set
# ------------------------------------------------------------

baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(X_train, y_train)

baseline_test_pred = baseline.predict(X_test)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_test_pred
)

baseline_precision = precision_score(
    y_test,
    baseline_test_pred,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_test_pred,
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    baseline_test_pred,
    zero_division=0
)

# Majority-class baseline has no useful ranking,
# so ROC-AUC is 0.5.
baseline_roc_auc = 0.5


# ------------------------------------------------------------
# 5. Save model comparison table
# ------------------------------------------------------------

comparison_results = pd.DataFrame([
    {
        "Model": "Majority Class Baseline",
        "Accuracy": baseline_accuracy,
        "Precision": baseline_precision,
        "Recall": baseline_recall,
        "F1": baseline_f1,
        "ROC_AUC": baseline_roc_auc
    },
    {
        "Model": "Logistic Regression",
        "Accuracy": lr_accuracy,
        "Precision": lr_precision,
        "Recall": lr_recall,
        "F1": lr_f1,
        "ROC_AUC": lr_roc_auc
    }
])

comparison_results.to_csv(
    output_dir / "logistic_regression_vs_baseline.csv",
    index=False
)

print("\n=== Test Set Comparison ===")
print(comparison_results.round(4))


# ------------------------------------------------------------
# 6. Save C tuning results
# ------------------------------------------------------------

if "tuning_results" in globals():

    tuning_results.to_csv(
        output_dir / "logistic_regression_C_tuning.csv",
        index=False
    )

    print("\nC tuning results saved.")

else:
    print("\nWARNING: tuning_results was not found.")


# ------------------------------------------------------------
# 7. Save threshold tuning results
# ------------------------------------------------------------

if "threshold_results" in globals():

    threshold_results.to_csv(
        output_dir / "logistic_regression_threshold_tuning.csv",
        index=False
    )

    print("Threshold tuning results saved.")

else:
    print("WARNING: threshold_results was not found.")


# ------------------------------------------------------------
# 8. Final Test Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    test_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Not Canceled",
        "Canceled"
    ]
)

fig, ax = plt.subplots(figsize=(7, 6))

disp.plot(
    ax=ax,
    values_format="d"
)

plt.title(
    "Logistic Regression - Test Confusion Matrix"
)

plt.tight_layout()

plt.savefig(
    output_dir / "logistic_regression_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nConfusion Matrix:")
print(cm)


# ------------------------------------------------------------
# 9. Save threshold tuning plot
# ------------------------------------------------------------

if "threshold_results" in globals():

    plt.figure(figsize=(9, 5))

    plt.plot(
        threshold_results["Threshold"],
        threshold_results["Precision"],
        marker="o",
        label="Precision"
    )

    plt.plot(
        threshold_results["Threshold"],
        threshold_results["Recall"],
        marker="s",
        label="Recall"
    )

    plt.plot(
        threshold_results["Threshold"],
        threshold_results["F1"],
        marker="^",
        label="F1"
    )

    plt.xlabel("Classification Threshold")
    plt.ylabel("Score")

    plt.title(
        "Validation Performance by Threshold"
    )

    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()

    plt.savefig(
        output_dir /
        "logistic_regression_threshold_tuning.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# ------------------------------------------------------------
# 10. Save C tuning plot
# ------------------------------------------------------------

if "tuning_results" in globals():

    plt.figure(figsize=(9, 5))

    plt.plot(
        tuning_results["C"].astype(str),
        tuning_results["Accuracy"],
        marker="o",
        label="Accuracy"
    )

    plt.plot(
        tuning_results["C"].astype(str),
        tuning_results["Precision"],
        marker="s",
        label="Precision"
    )

    plt.plot(
        tuning_results["C"].astype(str),
        tuning_results["Recall"],
        marker="^",
        label="Recall"
    )

    plt.plot(
        tuning_results["C"].astype(str),
        tuning_results["F1"],
        marker="D",
        label="F1"
    )

    plt.xlabel("C")
    plt.ylabel("Score")

    plt.title(
        "Logistic Regression Validation Performance by C"
    )

    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()

    plt.savefig(
        output_dir /
        "logistic_regression_C_tuning.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# ------------------------------------------------------------
# 11. Save trained Pipeline
# ------------------------------------------------------------

joblib.dump(
    model,
    output_dir /
    "logistic_regression_model.joblib"
)

print("\nModel saved successfully.")


# ------------------------------------------------------------
# 12. Save model configuration separately
# ------------------------------------------------------------

model_config = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Regularisation": "L2",
        "C": FINAL_C,
        "Threshold": FINAL_THRESHOLD,
        "Selection_metric": "Validation F1",
        "Train_rows": len(X_train),
        "Validation_rows": len(X_val),
        "Test_rows": len(X_test)
    }
])

model_config.to_csv(
    output_dir /
    "logistic_regression_configuration.csv",
    index=False
)


# ------------------------------------------------------------
# 13. Show all saved files
# ------------------------------------------------------------

print("\n=== Saved Files ===")

for file in sorted(output_dir.iterdir()):
    print(file)

In [ ]:
# ============================================================
# ERROR ANALYSIS DATASET
# ============================================================

error_df = X_test.copy()

error_df["actual"] = y_test.values
error_df["predicted"] = test_pred
error_df["cancel_probability"] = test_prob

def classify_error(row):
    if row["actual"] == 0 and row["predicted"] == 0:
        return "TN"
    elif row["actual"] == 0 and row["predicted"] == 1:
        return "FP"
    elif row["actual"] == 1 and row["predicted"] == 0:
        return "FN"
    else:
        return "TP"

error_df["error_type"] = error_df.apply(
    classify_error,
    axis=1
)

print(error_df["error_type"].value_counts())

In [ ]:
numeric_error_features = [
    "lead_time",
    "total_nights",
    "total_guests",
    "previous_cancellations",
    "previous_bookings_not_canceled"
]

error_summary = (
    error_df
    .groupby("error_type")[numeric_error_features]
    .mean()
    .round(2)
)

print(error_summary)

In [ ]:
hotel_error = pd.crosstab(
    error_df["hotel"],
    error_df["error_type"]
)

print(hotel_error)

In [ ]:
hotel_error_rate = (
    error_df
    .groupby("hotel")
    .apply(
        lambda x: pd.Series({
            "FP_rate": (
                (x["error_type"] == "FP").sum() / len(x)
            ),
            "FN_rate": (
                (x["error_type"] == "FN").sum() / len(x)
            ),
            "Total_error_rate": (
                x["actual"] != x["predicted"]
            ).mean()
        }),
        include_groups=False
    )
)

print(hotel_error_rate.round(4))

In [ ]:
deposit_error = pd.crosstab(
    error_df["deposit_type"],
    error_df["error_type"]
)

print(deposit_error)

In [ ]:
deposit_error_rate = (
    error_df
    .groupby("deposit_type")
    .apply(
        lambda x: pd.Series({
            "Count": len(x),
            "Error_rate": (
                x["actual"] != x["predicted"]
            ).mean(),
            "FP_rate": (
                x["error_type"] == "FP"
            ).mean(),
            "FN_rate": (
                x["error_type"] == "FN"
            ).mean()
        }),
        include_groups=False
    )
)

print(deposit_error_rate.round(4))

In [ ]:
# ============================================================
# ERROR ANALYSIS - LEAD TIME DISTRIBUTION
# ============================================================

import matplotlib.pyplot as plt
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

lead_time_by_error = [
    error_df.loc[
        error_df["error_type"] == label,
        "lead_time"
    ].dropna()
    for label in ["TN", "FP", "FN", "TP"]
]

plt.figure(figsize=(9, 5))

plt.boxplot(
    lead_time_by_error,
    tick_labels=["TN", "FP", "FN", "TP"],
    showfliers=False
)

plt.xlabel("Prediction Outcome")
plt.ylabel("Lead Time (days)")
plt.title("Lead Time Distribution by Prediction Outcome")

plt.tight_layout()

plt.savefig(
    output_dir / "error_analysis_lead_time.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Lead time error analysis plot saved successfully.")

In [ ]:
fp_examples = (
    error_df[
        error_df["error_type"] == "FP"
    ]
    .sort_values(
        "cancel_probability",
        ascending=False
    )
)

columns_to_show = [
    "hotel",
    "lead_time",
    "total_nights",
    "total_guests",
    "deposit_type",
    "market_segment",
    "customer_type",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "cancel_probability",
    "actual",
    "predicted"
]

print("=== Most Confident False Positives ===")

print(
    fp_examples[columns_to_show]
    .head(5)
    .to_string(index=False)
)

In [ ]:
fn_examples = (
    error_df[
        error_df["error_type"] == "FN"
    ]
    .sort_values(
        "cancel_probability",
        ascending=True
    )
)

print("\n=== Most Confident False Negatives ===")

print(
    fn_examples[columns_to_show]
    .head(5)
    .to_string(index=False)
)

In [ ]:
error_df.to_csv(
    output_dir / "logistic_regression_error_analysis.csv",
    index=False
)

error_summary.to_csv(
    output_dir / "logistic_regression_error_summary.csv"
)

fp_examples[columns_to_show].head(20).to_csv(
    output_dir / "logistic_regression_FP_examples.csv",
    index=False
)

fn_examples[columns_to_show].head(20).to_csv(
    output_dir / "logistic_regression_FN_examples.csv",
    index=False
)

print("Error analysis files saved.")

In [ ]:
# ============================================================
# ERROR RATE BY HOTEL TYPE
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

hotel_error_rate = (
    error_df
    .groupby("hotel")
    .apply(
        lambda x: pd.Series({
            "Overall Error Rate": (
                x["actual"] != x["predicted"]
            ).mean(),
            "False Positive Rate": (
                x["error_type"] == "FP"
            ).mean(),
            "False Negative Rate": (
                x["error_type"] == "FN"
            ).mean()
        }),
        include_groups=False
    )
)

print("=== Error Rate by Hotel Type ===")
print(hotel_error_rate.round(4))

# Save table
hotel_error_rate.to_csv(
    output_dir / "logistic_regression_hotel_error_rate.csv"
)

# Plot
plot_data = hotel_error_rate * 100

ax = plot_data.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.ylabel("Rate (%)")
plt.xlabel("Hotel Type")
plt.title("Logistic Regression Error Rates by Hotel Type")

plt.xticks(rotation=0)
plt.legend(title="Error Type")

plt.tight_layout()

plt.savefig(
    output_dir / "logistic_regression_hotel_error_rate.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nHotel error-rate figure saved successfully.")